[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eldanc/mlbootcamp2026/blob/main/lab_3_1_autoencoders.ipynb)


# UofT FASE ML Bootcamp
#### Wednesday, June 10, 2026
#### Autoencoders - Lab 1, Day 3
#### Teaching team: Teaching team: Eldan Cohen, Alex Olson, Nakul Upadhya, Hriday Chheda
##### Lab author: Alexander Olson, aolson@mie.utoronto.ca, edited by Jake Mosseri and Nakul Upadhya


We are going to learn about a _deep_ model for dimensionality reduction: Autoencoders.

An autoencoder is a special type of neural network with an unusual task: for some input X, all it has to do is return that input X as accurately as possible. But there's a catch, of course! Between the input and the output, the number of nodes in each hidden layer actually gets progressively _smaller_. This means that in the first half of the network, the network must learn how to represent the input in ever more compact formats. The second half does this in reverse, taking the smallest representation of the input and expanding it back out into the full, original data.

<img src="https://github.com/lyeskhalil/mlbootcamp/blob/master/img/ae.png?raw=1" alt="cross-val" width="500"/>

Before we build the neural network, we will first use this lab to briefly introduce two important ideas:
1. **MNIST**: the handwritten digit image dataset we will use throughout the lab.
2. **PCA**: a classical dimensionality reduction method that gives us a useful baseline before we move to autoencoders.

Autoencoders are much more powerful than PCA and t-SNE when it comes to learning compact representations of data, as we will see here. Let's first bring back PCA and give it an autoencoder's task: first, reduce the dimensionality of a dataset, and then recover the full dimensionality of the original input.

We will use MNIST dataset today, although through a slightly different mechanism to ease its compatibility with Pytorch. Run the code below to download the dataset:

In [ ]:
import torch
import torch.nn as nn
import torch.utils.data as Data
import torchvision
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
%matplotlib inline

In [ ]:
# Path parameters
MNIST_PATH = Path('./mnist/')
DOWNLOAD_MNIST = not MNIST_PATH.exists()

In [ ]:
train_data = torchvision.datasets.MNIST(
    root='./mnist/',
    train=True,                                     # this is training data
    transform=torchvision.transforms.ToTensor(),    # Converts a PIL.Image or numpy.ndarray to
                                                    # torch.FloatTensor of shape (C x H x W) and normalize in the range [0.0, 1.0]
    download=DOWNLOAD_MNIST,                        # download it if you don't have it
)

In [ ]:
print('Training data size:\t',train_data.data.size())     # (60000, 28, 28)
print('Testing data size:\t',train_data.targets.size())   # (60000)
plt.imshow(train_data.data[2].numpy(), cmap='gray')
plt.title('%i' % train_data.targets[2])
plt.show()

Before we use a neural network, let's try a simpler method: **Principal Component Analysis**, or **PCA**.

PCA is a dimensionality reduction method. It tries to find new directions in the data that capture as much variation as possible.

For MNIST, each image starts as 784 numbers:

```python
28 x 28 image  →  784 pixel values
```

PCA will try to compress those 784 numbers down to just 2 numbers:

```python
784 pixel values  →  2 PCA coordinates
```

Then we can try to reconstruct the original 784-pixel image from only those 2 numbers.

This is a very extreme compression. We should not expect the reconstruction to be perfect.

### Why PCA is useful here

PCA gives us a simple baseline for compression:

- It is **linear**.
- It is fast.
- It does not require neural networks.
- It gives us a useful comparison point for the autoencoder.

Later, the autoencoder will also compress each image to 2 numbers, but it will be allowed to learn a **nonlinear** representation.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

### Flatten the images

PCA expects a table where:

- each row is one example
- each column is one feature

For MNIST, that means each image becomes a row with 784 pixel features.

We will also scale pixel values from the range 0–255 to the range 0–1.


In [ ]:
# Convert from (60000, 28, 28) to (60000, 784)
X = train_data.data.numpy().reshape(-1, 28 * 28)

# Scale pixels from 0-255 to 0-1
X = X / 255.0

y = train_data.targets.numpy()

print('Original MNIST shape: ', train_data.data.shape)
print('Flattened data shape:', X.shape)
print('Labels shape:        ', y.shape)

### Fit PCA with 2 components

We now ask PCA to represent every image using only **two numbers**.

These two numbers are called the first two **principal components**.

Important: these components are not pixels. They are coordinates in a new 2D space learned from the dataset.


In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X)

print('Shape before PCA:', X.shape)
print('Shape after PCA: ', X_pca_2d.shape)
print()

Each point below is one MNIST image compressed into two numbers.

The colour shows the true digit label.

If PCA separates digits well, we should see clusters of similar colours. If the colours overlap, that means 2 linear dimensions are not enough to fully separate the digits.


In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_pca_2d[:, 0],
    X_pca_2d[:, 1],
    c=y,
    cmap=plt.colormaps.get_cmap('tab10'),
    s=5,
    alpha=0.5
)
plt.xlabel('Principal component 1')
plt.ylabel('Principal component 2')
plt.title('MNIST compressed to 2D using PCA')
plt.colorbar(scatter, label='Digit label')
plt.show()

### Reconstruct an image from only 2 PCA numbers

PCA also has an `inverse_transform` method.

This lets us go backwards:

```python
2 PCA coordinates  →  784 approximate pixel values
```

Then we reshape those 784 numbers back into a 28 by 28 image.

In [ ]:
# Reconstruct all images from their 2D PCA representation
X_reconstructed_from_2d = pca_2d.inverse_transform(X_pca_2d)

# Pick the same image shown above
example_index = 1
original_image = X[example_index].reshape(28, 28)
reconstructed_image = X_reconstructed_from_2d[example_index].reshape(28, 28)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].imshow(original_image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(reconstructed_image, cmap='gray')
axes[1].set_title('PCA reconstruction from 2 numbers')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### What if we use more PCA components?

Compressing to 2 dimensions is useful for visualization, but it throws away a lot of information.

**YOUR TURN**

Try changing the number of PCA components.

1. Fit PCA with `n_components=10`.
2. Reconstruct the same digit.
3. Compare it to the reconstruction using only 2 components.

Questions:

- Does the reconstructed digit look better?
- Why does using more components help?
- Why might 2 components still be useful even if reconstruction quality is poor?

In [ ]:
#TODO: ADD your code here

Let's now move on to building an autoencoder for the same task. Autoencoders are easy networks to build, split into the _encoder_, which 'steps' the data down to the final compact representation, and the _decoder_, which is a mirror image of the encoder.


In [ ]:
myEncoder = nn.Sequential(
    nn.Linear(28*28, 128),
    nn.Tanh(),
    nn.Linear(128, 64),
    nn.Tanh(),
    nn.Linear(64, 12),
    nn.Tanh(),
    nn.Linear(12, 2),
    nn.Tanh(),
)

As you may expect, designing the structure of the encoder is something of an art, and it requires balance between the time and input data required to train the network, and performance. Here we are using four step-down operations (the linear layers), which take us from the input of size 784 down to just two dimensions at the bottom. Between each step-down layer is a non-linear activation layer.

---

**Your turn**

It would of course be possible to go straight from the input size to the final number of dimensions, but we would lose an incredibly important aspect of neural networks in doing so.

* What would we miss out on? ______
* Why is this a problem? ______

In the cell below, build the structure of the decoder layer for our network. Remember, this is a mirror image of our encoder!

In [ ]:
myDecoder = nn.Sequential(
    nn.Linear(2, 12),
    nn.Tanh(),
    nn.Linear(12, 64),
    nn.Tanh(),
    nn.Linear(64, 128),
    nn.Tanh(),
    nn.Linear(128, 28*28),
)

---

Finally, we just need some boilerplate class code to bring the whole thing together into a PyTorch network:

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()

        self.encoder = myEncoder
        self.decoder = myDecoder

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

Now let's create an instance of our network. We also need to define a few other parameters, like the loss function and the optimizer.

For the loss function, we will be using Mean Squared Error, which we covered in the second lab. For the optimizer, let's use an advanced optimizer called Adam:

In [ ]:
autoencoder = AutoEncoder()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.005)
loss_func = nn.MSELoss()

We'll use a helper function during training which will pass us the data as we go. It's important to remember that for an autoencoder, the input and the label are identical, so we don't have any labels per se.

In [ ]:
train_loader = Data.DataLoader(dataset=train_data, batch_size=64, shuffle=True)

Our training function is going to show us the recovered images at the end of each epoch, to help us get an idea of how the training process is going. Beforehand, we will just plot out five of the digits so we can compare our autoencoder's output to what the target looks like:

In [ ]:
N_TEST_IMG = 5
# initialize figure
f, a = plt.subplots(1, N_TEST_IMG, figsize=(5, 2))
# original data (first row) for viewing
view_data = train_data.data[:N_TEST_IMG].view(-1, 28*28).type(torch.FloatTensor)/255.
for i in range(N_TEST_IMG):
    a[i].imshow(np.reshape(view_data.data.numpy()[i], (28, 28)), cmap='gray'); a[i].set_xticks(()); a[i].set_yticks(())

OK, now we're ready to train! Before running this code, make sure you understand what is happening between the dashed lines (the rest is just there to draw the graphs, so it's not important). The comments should help. Once you feel comfortable, run the cell! It will take a little while to complete, but you will get progress updates as it goes.

In [ ]:
for epoch in range(25):
    for step, (x, _) in enumerate(train_loader):
        #----------------------------------------------------------------------------#
        b_x = x.view(-1, 28*28)   # batch x, shape (batch, 28*28)
        b_y = x.view(-1, 28*28)   # batch y, shape (batch, 28*28)

        encoded, decoded = autoencoder(b_x)

        loss = loss_func(decoded, b_y)      # mean square error
        optimizer.zero_grad()               # clear gradients for this training step
        loss.backward()                     # backpropagation, compute gradients
        optimizer.step()                    # apply gradients
        #----------------------------------------------------------------------------#
    print('Epoch: ', epoch, '| train loss: %.4f' % loss.data.numpy())
    f, a = plt.subplots(1, N_TEST_IMG, figsize=(5, 2))

    # plotting decoded image (second row)
    _, decoded_data = autoencoder(view_data)
    for i in range(N_TEST_IMG):
        a[i].clear()
        a[i].imshow(np.reshape(decoded_data.data.numpy()[i], (28, 28)), cmap='gray')
        a[i].set_xticks(()); a[i].set_yticks(())
    plt.draw(); plt.pause(0.1)

The third image in the column is our Autoencoder's recovered version of the number 4 we looked at with PCA. How does it look? Could it be better?

We can also plot the encoded data in the same way we did for PCA and t-SNE above:

In [ ]:
view_data = train_data.data.view(-1, 28*28).type(torch.FloatTensor)/255.
encoded_data, _ = autoencoder(view_data)
X, Y = encoded_data.data[:, 0].numpy(), encoded_data.data[:, 1].numpy()
values = train_data.targets.numpy()

plt.figure(figsize=(16,10))
plt.scatter(X, Y,
            c=values, edgecolor='none', alpha=0.5,
            cmap=plt.colormaps.get_cmap('tab10'))
plt.xlabel('component 1')
plt.ylabel('component 2')
plt.colorbar();

Now we are going to plot different outputs based on changes to the representation (encoded) space. We will loop through all different combinations between -1 and 1 and pass it through the decoder to see what the autoencoder has learned.

In [ ]:
from torch import Tensor
f, a = plt.subplots(9, 9, figsize=(12, 12))
for i,v in enumerate([1,0.75,0.5,0.25,0,-0.25,-0.5,-0.75,-1]):
  for j,k in enumerate([-1,-0.75,-0.5,-0.25,0,0.25,0.5,0.75,1]):
    a[i,j].imshow(np.reshape(
        autoencoder.decoder(Tensor([k,v])).detach().numpy(),
        (28, 28)), cmap='gray'); a[i,j].set_xticks(()); a[i,j].set_yticks(())